In [0]:
USE demo_snowday;

In [0]:
CREATE TABLE workspace.demo_snowday.landing_employee_cdf (
  id STRING,
  fullname STRING,
  sal STRING,
  doj STRING
) USING delta
TBLPROPERTIES (delta.enableChangeDataFeed = true);

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-5580768949261650>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', 'CREATE TABLE workspace.demo_snowday.landing_employee_cdf (\n  id STRING,\n  fullname STRING,\n  sal STRING,\n  doj STRING\n) USING delta\nTBLPROPERTIES (delta.enableChangeDataFeed = true);\n')

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2541, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2539 with self.builtin_trap:
   2540     args = (magic_arg_s, cell)
-> 2541     result = fn(*args, **kwargs)
   2543 # The code below prevents the output from being displayed
   2544 # when using magics with decorator @output_can_be_silenced
   2545 # when the last Python token in the expression is a ';'.
   2546 if getattr(fn, magic.MAGIC_OUTPUT_CAN_BE_SILENCED, False):

File /databricks/python_shell/

In [0]:
INSERT INTO workspace.demo_snowday.landing_employee_cdf VALUES
('11','Adam','3000','2020-05-01'),
('13','Bob','200','2018-05-25'),
('14','King','100','2019-05-01');

num_affected_rows,num_inserted_rows
3,3


In [0]:
UPDATE workspace.demo_snowday.landing_employee_cdf
SET sal = '260'
WHERE id = '13';

num_affected_rows
1


In [0]:
UPDATE workspace.demo_snowday.landing_employee_cdf
SET sal = '275'
WHERE id = '13';

num_affected_rows
1


In [0]:
-- BRONZE should show multiple change events
SELECT id, _change_type, _commit_version, _commit_timestamp, sal
FROM workspace.demo_snowday.bronze_employee_cdf
WHERE id = '13'
ORDER BY _commit_version, _commit_timestamp;

id,_change_type,_commit_version,_commit_timestamp,sal
13,insert,4,2026-03-02T20:47:56.000Z,250
13,update_postimage,5,2026-03-02T21:20:18.000Z,260
13,update_preimage,5,2026-03-02T21:20:18.000Z,250
13,update_preimage,6,2026-03-02T21:20:30.000Z,260
13,update_postimage,6,2026-03-02T21:20:30.000Z,275


In [0]:
-- 2) STAGE should show multiple versions
SELECT id, sal, __START_AT, __END_AT
FROM workspace.demo_snowday.stage_employee_cdf_type2
WHERE id = 13
ORDER BY __START_AT;

id,sal,__START_AT,__END_AT
13,250.0,"List(2026-03-02T20:47:56.000Z, 4)","List(2026-03-02T21:20:18.000Z, 5)"
13,260.0,"List(2026-03-02T21:20:18.000Z, 5)","List(2026-03-02T21:20:30.000Z, 6)"
13,275.0,"List(2026-03-02T21:20:30.000Z, 6)",null


In [0]:
-- 3) GOLD must show show Active + Inactive
SELECT row_status, COUNT(*) AS cnt
FROM workspace.demo_snowday.gold_employee_cdf_type2
GROUP BY row_status
ORDER BY row_status;

row_status,cnt
A,3
I,2


In [0]:
-- 2) Requirement: Eliminate CDF common schema attributes from GOLD
-- 2A) Quick visual check of columns
SHOW COLUMNS IN workspace.demo_snowday.gold_employee_cdf_type2;

col_name
id
fullname
sal
doj
start_date
end_date
row_status


In [0]:
-- 2B) Programmatic “should return 0 rows” check
SELECT column_name
FROM workspace.information_schema.columns
WHERE table_schema = 'demo_snowday'
  AND table_name = 'gold_employee_cdf_type2'
  AND lower(column_name) IN (
    '_change_type','_commit_version','_commit_timestamp',
    'cdf_change_type','cdf_commit_version','cdf_commit_timestamp'
  );


-- Validate Active (A) / Inactive (I) Logic
SELECT *
FROM workspace.demo_snowday.gold_employee_cdf_type2
WHERE
      (end_date IS NULL AND row_status <> 'A')
   OR (end_date IS NOT NULL AND row_status <> 'I');

id,fullname,sal,doj,start_date,end_date,row_status


In [0]:
-- verify start_date and end_date Exist
SELECT column_name, data_type
FROM workspace.information_schema.columns
WHERE table_schema = 'demo_snowday'
  AND table_name = 'gold_employee_cdf_type2'
  AND lower(column_name) IN ('start_date','end_date');

column_name,data_type
start_date,DATE
end_date,DATE


In [0]:
-- Verify __start* / __end* DO NOT Exist
SELECT column_name
FROM workspace.information_schema.columns
WHERE table_schema = 'demo_snowday'
  AND table_name = 'gold_employee_cdf_type2'
  AND lower(column_name) LIKE '__start%'
   OR lower(column_name) LIKE '__end%';

column_name
__END_AT
__END_AT


In [0]:
-- Verify DATE columns contain only date
SELECT *
FROM workspace.demo_snowday.gold_employee_cdf_type2
WHERE start_date != CAST(start_date AS DATE)
   OR (end_date IS NOT NULL AND end_date != CAST(end_date AS DATE));

id,fullname,sal,doj,start_date,end_date,row_status


In [0]:
-- 3) Requirement: rename __start_date/__end_date → start_date/end_date
-- AND if new columns were created, do NOT include __start/__end in GOLD
-- 3A) Confirm start_date and end_date exist
SELECT column_name
FROM workspace.information_schema.columns
WHERE table_schema = 'demo_snowday'
  AND table_name = 'gold_employee_cdf_type2'
  AND lower(column_name) IN ('start_date','end_date');


column_name
start_date
end_date


In [0]:
-- 3B) Confirm __start* / __end* do NOT exist in GOLD
SELECT lower(column_name) AS col_name
FROM information_schema.columns
WHERE table_catalog = 'workspace'
  AND table_schema = 'demo_snowday'
  AND table_name = 'gold_employee_cdf_type2'
  AND (
        lower(column_name) LIKE '__start%' 
     OR lower(column_name) LIKE '__end%'
      );

col_name


In [0]:
-- 4) Requirement: date columns are datetime datatype but contain only date
-- 4A) Confirm datatype is DATE
DESCRIBE workspace.demo_snowday.gold_employee_cdf_type2;

-- 4B) Extra validation: casting doesn’t change values (means no time component)
SELECT *
FROM workspace.demo_snowday.gold_employee_cdf_type2
WHERE
  CAST(start_date AS DATE) <> start_date
  OR (end_date IS NOT NULL AND CAST(end_date AS DATE) <> end_date);


format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,6ab49809-148c-4f8b-bf6c-491b0406a99d,workspace.demo_snowday.landing_employee_cdf,null,,2026-03-03T03:13:48.362Z,2026-03-03T03:27:24.000Z,List(),List(),1,2253,"Map(delta.parquet.compression.codec -> zstd, delta.enableChangeDataFeed -> true, delta.enableDeletionVectors -> true, delta.enableRowTracking -> true, delta.rowTracking.materializedRowCommitVersionColumnName -> _row-commit-version-col-a78a883a-44f8-4104-9e3b-7d93f4f2900b, delta.rowTracking.materializedRowIdColumnName -> _row-id-col-5fe6727f-f973-420d-b270-6b29ec171ee1)",3,7,"List(appendOnly, changeDataFeed, deletionVectors, domainMetadata, invariants, rowTracking)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
-- 5) Requirement: row_status maintained as Active (A) / Inactive (I)
-- 5A) Validate rule alignment
-- Active means end_date IS NULL, inactive means end_date IS NOT NULL.
SELECT
  row_status,
  COUNT(*) AS cnt
FROM workspace.demo_snowday.gold_employee_cdf_type2
GROUP BY row_status
ORDER BY row_status;

row_status,cnt
A,3
I,2


In [0]:
-- validate no mismatches
SELECT *
FROM workspace.demo_snowday.gold_employee_cdf_type2
WHERE
  (end_date IS NULL AND row_status <> 'A')
  OR (end_date IS NOT NULL AND row_status <> 'I');

id,fullname,sal,doj,start_date,end_date,row_status


In [0]:
-- First confirm whether STAGE (SCD2 table) has history. If STAGE has only one row too, the issue is upstream (apply_changes / sequence / deletes).
SELECT id,
       COUNT(*) AS versions,
       MIN(__START_AT) AS min_start,
       MAX(__START_AT) AS max_start
FROM workspace.demo_snowday.stage_employee_cdf_type2
WHERE id = 13
GROUP BY id;


id,versions,min_start,max_start
13,3,"List(2026-03-02T20:47:56.000Z, 4)","List(2026-03-02T21:20:30.000Z, 6)"


In [0]:
-- 6) Sanity check: SCD2 behavior for updated/deleted keys
-- 6A) Updated key should have 2 versions (old ended + new active)
SELECT id, fullname, sal, start_date, end_date, row_status
FROM workspace.demo_snowday.gold_employee_cdf_type2
WHERE id = 13
ORDER BY start_date, end_date;

SELECT id, _change_type, _commit_version, _commit_timestamp, sal
FROM workspace.demo_snowday.bronze_employee_cdf
WHERE id = '13'
ORDER BY _commit_version, _commit_timestamp;

id,_change_type,_commit_version,_commit_timestamp,sal
13,insert,4,2026-03-02T20:47:56.000Z,250
13,update_postimage,5,2026-03-02T21:20:18.000Z,260
13,update_preimage,5,2026-03-02T21:20:18.000Z,250
13,update_preimage,6,2026-03-02T21:20:30.000Z,260
13,update_postimage,6,2026-03-02T21:20:30.000Z,275


In [0]:
-- 2) Check whether the update actually produced a CDF update event
-- Since i am using CDF, the source table must emit an update_* event.
-- Run this on the landing table to confirm the change happened:
SELECT * 
FROM workspace.demo_snowday.landing_employee_cdf
WHERE id = '13';

-- checking whether CDF is enabled 
DESCRIBE DETAIL workspace.demo_snowday.landing_employee_cdf;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,6ab49809-148c-4f8b-bf6c-491b0406a99d,workspace.demo_snowday.landing_employee_cdf,null,,2026-03-03T03:13:48.362Z,2026-03-03T03:27:24.000Z,List(),List(),1,2253,"Map(delta.parquet.compression.codec -> zstd, delta.enableChangeDataFeed -> true, delta.enableDeletionVectors -> true, delta.enableRowTracking -> true, delta.rowTracking.materializedRowCommitVersionColumnName -> _row-commit-version-col-a78a883a-44f8-4104-9e3b-7d93f4f2900b, delta.rowTracking.materializedRowIdColumnName -> _row-id-col-5fe6727f-f973-420d-b270-6b29ec171ee1)",3,7,"List(appendOnly, changeDataFeed, deletionVectors, domainMetadata, invariants, rowTracking)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false
